# LightGBM v2.2 — DNS Attack Detection

## Why LightGBM?
- **Fastest** of the three models (histogram-based, leaf-wise growth)
- Native categorical feature support — encodes `protocol` directly
- `is_unbalance=True` handles BENIGN/ATTACK imbalance automatically
- Better with large datasets than XGBoost

## Same Pipeline as XGBoost v2.2
- Log-transform 8 skewed features
- `is_unbalance=True` (LGBM equivalent of `scale_pos_weight`)
- F-beta threshold tuning (beta=2.0 → prioritise attack recall)
- Balance report showing both BENIGN and ATTACK recall

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, fbeta_score
)
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('ggplot')
print(f'[OK] LightGBM version: {lgb.__version__}')

## 1. Load Data

In [ ]:
FILE_PATH = r"C:\Users\shenal\Downloads\reseraach\PCAPS_Used\Final_Balanced_Attack_and_Benign\Final_balanced_Attack_and_Benign_new_Shuffled.csv"

print(f'[INFO] Loading: {FILE_PATH}')
df = pd.read_csv(FILE_PATH)
print(f'[OK]  Rows: {len(df):,}   Cols: {len(df.columns)}')
print(f'\nClass Distribution:')
print(df['label'].value_counts())
df.head(3)

## 2. Preprocessing + Log-Transform

In [ ]:
COLS_TO_DROP = ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol_number']
df = df.drop(columns=COLS_TO_DROP, errors='ignore')

df.replace([np.inf, -np.inf], 0, inplace=True)
df.fillna(0, inplace=True)

LABEL_MAP = {'BENIGN': 0, 'ATTACK': 1}
df['label'] = df['label'].str.upper().map(LABEL_MAP).fillna(0).astype(int)
print(f'Labels: {dict(df["label"].value_counts())}')

# LightGBM can handle categoricals natively — just mark it as category dtype
PROTOCOL_CLASSES = ['DOH', 'DOT', 'TRADITIONAL', 'UNKNOWN', 'TCP', 'UDP']
df['protocol'] = df['protocol'].astype(str).apply(
    lambda x: x if x in PROTOCOL_CLASSES else 'UNKNOWN'
)
df['protocol'] = df['protocol'].astype('category')
print(f'Protocol encoded as category: {df["protocol"].cat.categories.tolist()}')

# Log-transform skewed features
LOG_FEATURES = [
    'bwd_packets_per_sec',
    'flow_bytes_per_sec',
    'flow_packets_per_sec',
    'fwd_packets_per_sec',
    'dns_queries_per_second',
    'total_fwd_packets',
    'total_bwd_packets',
    'dns_amplification_factor',
]
print('\n[LOG-TRANSFORM] Compressing skewed features...')
for col in LOG_FEATURES:
    if col in df.columns:
        df[col] = np.log1p(df[col].clip(lower=0))
        print(f'  log1p({col})')

print('\n[OK] Preprocessing complete')

## 3. Train / Test Split

In [ ]:
X = df.drop(columns=['label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
n_benign = (y_train == 0).sum()
n_attack = (y_train == 1).sum()
print(f'Train: BENIGN={n_benign:,}  ATTACK={n_attack:,}  ratio={n_attack/n_benign:.2f}')
print(f'Test:  {X_test.shape}')
print(f'Categorical features: {X.select_dtypes("category").columns.tolist()}')

## 4. LightGBM Training

### Key LGBM Parameters
| Parameter | Value | Effect |
|---|---|---|
| `is_unbalance` | `True` | Auto-handles BENIGN/ATTACK class imbalance |
| `num_leaves` | `63` | Controls tree complexity (higher=more complex) |
| `feature_fraction` | `0.7` | Use 70% features per tree — prevents bwd_pps dominance |
| `bagging_fraction` | `0.8` | Row subsampling per tree |
| `min_child_samples`| `20` | Min samples per leaf — prevents overfit |
| `reg_lambda` | `1.0` | L2 regularisation |

**Tuning Guide**:
- Attacks missed → set `is_unbalance=False` and use `class_weight={0:1, 1:2}`
- Too many FP on benign → reduce `num_leaves` to `31`

In [ ]:
model = lgb.LGBMClassifier(
    # Core
    objective          = 'binary',
    metric             = ['binary_logloss', 'auc'],
    
    # Class balance
    is_unbalance       = True,    # Auto-weight BENIGN vs ATTACK
    
    # Tree architecture
    n_estimators       = 800,
    learning_rate      = 0.03,
    num_leaves         = 63,      # 2^max_depth - 1; 63 = ~depth 6
    max_depth          = -1,      # -1 = no limit (num_leaves controls complexity)
    min_child_samples  = 20,
    
    # Regularisation
    feature_fraction   = 0.7,     # % features per tree
    bagging_fraction   = 0.8,
    bagging_freq       = 5,
    reg_alpha          = 0.05,
    reg_lambda         = 1.0,
    
    # Speed
    n_jobs             = -1,
    random_state       = 42,
    verbose            = -1,
)

# LightGBM needs categorical feature names explicitly
cat_features = X_train.select_dtypes('category').columns.tolist()
print(f'[TRAIN] Categorical features: {cat_features}')

model.fit(
    X_train, y_train,
    categorical_feature = cat_features,
    eval_set            = [(X_test, y_test)],
    callbacks           = [lgb.log_evaluation(100)],
)
print('[DONE] Training complete.')

## 5. Threshold Tuning (ATTACK Recall Priority)

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

BETA = 2.0   # Prioritise attack recall

thresholds = np.arange(0.05, 0.95, 0.01)
best_thresh, best_score = 0.5, 0

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    score = fbeta_score(y_test, y_pred_t, beta=BETA, zero_division=0)
    if score > best_score:
        best_score, best_thresh = score, t

print(f'[TUNING] Optimal threshold = {best_thresh:.2f}  (F-{BETA} = {best_score:.4f})')

scores = [fbeta_score(y_test, (y_prob >= t).astype(int), beta=BETA, zero_division=0)
          for t in thresholds]
plt.figure(figsize=(10, 4))
plt.plot(thresholds, scores, color='darkorange')
plt.axvline(best_thresh, color='red',  linestyle='--', label=f'Best={best_thresh:.2f}')
plt.axvline(0.5,         color='gray', linestyle=':',  label='Default=0.50')
plt.xlabel('Threshold'); plt.ylabel(f'F-beta (beta={BETA})')
plt.title('LightGBM — Threshold vs Score'); plt.legend(); plt.show()

## 6. Balance Report

In [ ]:
y_pred = (y_prob >= best_thresh).astype(int)

print(f'=== CLASSIFICATION REPORT (threshold={best_thresh:.2f}) ===')
print(classification_report(y_test, y_pred, target_names=['BENIGN','ATTACK'], digits=4))

auc = roc_auc_score(y_test, y_prob)
print(f'ROC-AUC: {auc:.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['BENIGN','ATTACK'], yticklabels=['BENIGN','ATTACK'])
plt.title(f'LightGBM Confusion Matrix @ threshold={best_thresh:.2f}')
plt.ylabel('Actual'); plt.xlabel('Predicted'); plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
benign_recall  = tn / (tn + fp) if (tn + fp) > 0 else 0
attack_recall  = tp / (tp + fn) if (tp + fn) > 0 else 0
false_pos_rate = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f'\n===============================')
print(f' LGBM BALANCE REPORT')
print(f'===============================')
print(f' BENIGN Recall  : {benign_recall*100:6.1f}%')
print(f' ATTACK Recall  : {attack_recall*100:6.1f}%')
print(f' False Pos Rate : {false_pos_rate*100:6.1f}%')
print(f'===============================')

if attack_recall < 0.85:
    print('\n[ADVICE] Attack recall low. Try: is_unbalance=False + class_weight={0:1, 1:2}')
elif benign_recall < 0.85:
    print('\n[ADVICE] Too many FP. Try: reduce num_leaves to 31 or increase min_child_samples')
else:
    print('\n[GOOD] Both recalls above 85%!')

## 7. Feature Importance (LGBM Split-based)

In [ ]:
# LightGBM provides two importance types:
#   'split'  = how many times a feature is used to split
#   'gain'   = total gain from splits (more meaningful for DNS features)
for imp_type in ['split', 'gain']:
    importance = pd.Series(
        model.booster_.feature_importance(importance_type=imp_type),
        index=X.columns
    )
    top15 = importance.nlargest(15)
    plt.figure(figsize=(10, 6))
    top15.sort_values().plot(kind='barh', color='darkorange')
    plt.title(f'Top 15 Feature Importances — LightGBM ({imp_type})')
    plt.xlabel(f'Importance ({imp_type})')
    plt.tight_layout(); plt.show()
    print(f'\nTop 5 by {imp_type}:')
    for feat, val in top15.head(5).items():
        print(f'  {feat:<35} {val:.1f}')

bwd_gain = pd.Series(
    model.booster_.feature_importance(importance_type='gain'),
    index=X.columns
).get('bwd_packets_per_sec', 0)
print(f'\n[CHECK] bwd_packets_per_sec gain = {bwd_gain:.1f}')

## 8. Save Model Bundle

In [ ]:
bundle = {
    'model':            model,
    'threshold':        best_thresh,
    'log_features':     LOG_FEATURES,
    'protocol_classes': PROTOCOL_CLASSES,
    'model_type':       'lightgbm',
    'cat_features':     cat_features,
    'beta':             BETA,
}

with open('lgbm_model_v2.2.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print(f'[SAVED] lgbm_model_v2.2.pkl')
print(f'  threshold    = {best_thresh:.2f}')
print(f'  ATTACK Recall = {attack_recall*100:.1f}%')
print(f'  BENIGN Recall = {benign_recall*100:.1f}%')
print(f'  ROC-AUC       = {auc:.4f}')